# CS771 Minor 2: MAP Inference for MNIST Digit Reconstruction

This notebook demonstrates a single-step Maximum A Posteriori (MAP) inference framework over multivariate Gaussian densities to reconstruct missing pixels (censoring) in MNIST digits.

In [ ]:
import numpy as np
from numpy import linalg as lin
import struct
from array import array
from os.path import join
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import time
import gc

### MNIST Data Loader (unchanged from lecture code)

In [ ]:
class MnistDataloader(object):
    def __init__(self, training_images_filepath, training_labels_filepath,
                 test_images_filepath, test_labels_filepath):
        self.training_images_filepath = training_images_filepath
        self.training_labels_filepath = training_labels_filepath
        self.test_images_filepath = test_images_filepath
        self.test_labels_filepath = test_labels_filepath

    def read_images_labels(self, images_filepath, labels_filepath):
        labels = []
        with open(labels_filepath, 'rb') as file:
            magic, size = struct.unpack(">II", file.read(8))
            if magic != 2049:
                raise ValueError('Magic number mismatch, expected 2049, got {}'.format(magic))
            labels = array("B", file.read())

        with open(images_filepath, 'rb') as file:
            magic, size, rows, cols = struct.unpack(">IIII", file.read(16))
            if magic != 2051:
                raise ValueError('Magic number mismatch, expected 2051, got {}'.format(magic))
            image_data = array("B", file.read())
        images = []
        for i in range(size):
            images.append([0] * rows * cols)
        for i in range(size):
            img = np.array(image_data[i * rows * cols:(i + 1) * rows * cols])
            img = img.reshape(28, 28)
            images[i][:] = img

        return images, labels

    def load_data(self):
        x_train, y_train = self.read_images_labels(self.training_images_filepath, self.training_labels_filepath)
        x_test, y_test = self.read_images_labels(self.test_images_filepath, self.test_labels_filepath)
        return (x_train, y_train), (x_test, y_test)

### Helper functions (unchanged from lecture code)

In [ ]:
def flattenTensor(X):
    n = X.shape[0]
    d = np.prod(X.shape[1:])
    return X.reshape(n, d)

def truncatePixels(X, low=0, high=1):
    X[X < low] = low
    X[X > high] = high
    return X

### Training (unchanged from lecture code)

In [ ]:
def learnClassConditionalDist(X, y):
    (labels, labelCounts) = np.unique(y, return_counts=True)
    C = labels.size
    (n, d) = X.shape
    muVals = np.zeros((C, d))
    SigmaVals = np.zeros((C, d, d))
    for c in range(C):
        XThisLabel = X[y == c]
        muVals[c] = np.mean(XThisLabel, axis=0)
        XCent = XThisLabel - muVals[c]
        SigmaVals[c] = 1 / labelCounts[c] * (XCent.T).dot(XCent)
    return (C, list(zip(muVals, SigmaVals, np.arange(C), labelCounts / n)))

### Two-step Inference (lecture code - unchanged)

In [ ]:
def predictClassScores(X, mu, Sigma, p, mask=[]):
    if len(mask) == 0:
        mask = np.ones((X.shape[1],), dtype=bool)
    XCent = X[:, mask] - mu[mask]
    SInv = lin.pinv(Sigma[mask, :][:, mask])
    (sign, logdet) = lin.slogdet(Sigma[mask, :][:, mask])
    if sign <= 0:
        SLogDet = 0
    else:
        SLogDet = logdet
    term1 = 0.5 * (-SLogDet - np.sum(np.multiply(np.dot(XCent, SInv), XCent), axis=1))
    term2 = np.log(p)
    return term1 + term2

def predictGen(X, model, C, mask=[]):
    classScores = np.zeros((X.shape[0], C))
    for mu, Sigma, c, p in model:
        classScores[:, int(c)] = predictClassScores(X, mu, Sigma, p, mask)
    return np.argmax(classScores, axis=1)

def reconstructImages(X, yPred, model, mask):
    XRecon = np.zeros(X.shape)
    XRecon[:, mask] = X[:, mask]
    for i in range(X.shape[0]):
        (mu, Sigma, c, p) = model[yPred[i]]
        recon = mu[~mask] + Sigma[~mask, :][:, mask].dot(
            lin.pinv(Sigma[mask, :][:, mask]).dot((XRecon[i, mask] - mu[mask]))
        )
        XRecon[i, ~mask] = recon
    return truncatePixels(XRecon)

### Single-step (Joint) Inference - Our new method

In [ ]:
def robustLogDet(M):
    """Compute log pseudo-determinant (product of non-zero eigenvalues)."""
    sign, logdet = lin.slogdet(M)
    if sign > 0:
        return logdet
    eigvals = np.linalg.eigvalsh(M)
    pos_eigvals = eigvals[eigvals > 1e-10]
    if len(pos_eigvals) > 0:
        return np.sum(np.log(pos_eigvals))
    return 0.0

def jointPredictReconstruct(X, model, C, mask=[]):
    """
    Single-step inference: jointly predict class and reconstruct.
    Score for class c: -0.5 * ln|Sigma_c| - 0.5 * (x_o - mu_o)^T Sigma_oo^{-1} (x_o - mu_o) + ln P[y=c]
    Reconstruction: x_m = mu_m + Sigma_mo * Sigma_oo^{-1} * (x_o - mu_o)
    """
    if len(mask) == 0:
        mask = np.ones((X.shape[1],), dtype=bool)

    n = X.shape[0]
    d = X.shape[1]
    classScores = np.zeros((n, C))

    classData = []
    for mu, Sigma, c, p in model:
        c = int(c)
        Sigma_oo = Sigma[mask, :][:, mask]
        SInv_oo = lin.pinv(Sigma_oo)

        fullLogDet = robustLogDet(Sigma)

        XCent = X[:, mask] - mu[mask]
        mahal = np.sum(np.multiply(np.dot(XCent, SInv_oo), XCent), axis=1)
        classScores[:, c] = 0.5 * (-fullLogDet - mahal) + np.log(p)

        reconMatrix = Sigma[~mask, :][:, mask].dot(SInv_oo)
        classData.append((c, mu, reconMatrix, XCent))

    yPred = np.argmax(classScores, axis=1)

    XRecon = np.zeros_like(X)
    XRecon[:, mask] = X[:, mask]
    for c, mu, reconMatrix, XCent in classData:
        idx = np.where(yPred == c)[0]
        if len(idx) > 0:
            XRecon[idx[:, None], ~mask] = mu[~mask] + reconMatrix.dot(XCent[idx].T).T

    return yPred, truncatePixels(XRecon)

### Censoring function (parameterized)

In [ ]:
def censorImages(X, start, end):
    """Censor the central portion X[:, start:end, start:end] = 0"""
    Xc = X.copy()
    Xc[:, start:end, start:end] = 0
    return Xc

def getMask(start, end, imShape=(28, 28)):
    """Get bool mask for the flat image: True = observed, False = missing"""
    template = np.ones(imShape)
    template[start:end, start:end] = 0
    return template.flatten() == 1

### Main Experiment

In [ ]:
if __name__ == "__main__":
    print("Loading MNIST data...")
    input_path = 'MNIST/archive'
    training_images_filepath = join(input_path, 'train-images-idx3-ubyte/train-images-idx3-ubyte')
    training_labels_filepath = join(input_path, 'train-labels-idx1-ubyte/train-labels-idx1-ubyte')
    test_images_filepath = join(input_path, 't10k-images-idx3-ubyte/t10k-images-idx3-ubyte')
    test_labels_filepath = join(input_path, 't10k-labels-idx1-ubyte/t10k-labels-idx1-ubyte')

    mnist_dataloader = MnistDataloader(training_images_filepath, training_labels_filepath,
                                        test_images_filepath, test_labels_filepath)
    ((XTrain, yTrain), (XTest, yTest)) = mnist_dataloader.load_data()
    yTrain = np.array(yTrain)
    yTest = np.array(yTest)

    tmp = np.zeros((len(XTrain), 28, 28))
    for i in range(len(XTrain)):
        tmp[i, :] = np.concatenate(XTrain[i]).reshape((28, 28))
    XTrain = tmp

    tmp = np.zeros((len(XTest), 28, 28))
    for i in range(len(XTest)):
        tmp[i, :] = np.concatenate(XTest[i]).reshape((28, 28))
    XTest = tmp

    imShape = XTrain.shape[1:]
    XTrainFlat = flattenTensor(XTrain / 256)
    XTestFlat_orig = flattenTensor(XTest / 256)

    print("Training class-conditional Gaussians...")
    (C, model) = learnClassConditionalDist(XTrainFlat, yTrain)

    yPredFull = predictGen(XTestFlat_orig, model, C)
    fullAcc = np.sum(yPredFull == yTest) / yTest.size
    print(f"Full image accuracy: {fullAcc:.4f}")

    censoring_levels = [
        (4, 24, "51%"),
        (6, 22, "32%"),
        (8, 20, "18%"),
        (10, 18, "8%"),
        (12, 16, "2%"),
    ]

    two_step_accs = []
    single_step_accs = []
    two_step_times = []
    single_step_times = []
    missing_pcts = []

    for (start, end, pct_label) in censoring_levels:
        print(f"\n--- Censoring: X[:, {start}:{end}, {start}:{end}] = 0 ({pct_label} missing) ---")

        XTestCensored = censorImages(XTest, start, end)
        XTestCensorFlat = flattenTensor(XTestCensored / 256)
        mask = getMask(start, end)
        missing_pcts.append(pct_label)

        gc.collect()
        gc.disable()
        tic = time.perf_counter()
        yPred2 = predictGen(XTestCensorFlat, model, C, mask)
        toc = time.perf_counter()
        gc.enable()
        t2_time = toc - tic
        t2_acc = np.sum(yPred2 == yTest) / yTest.size
        two_step_accs.append(t2_acc)
        two_step_times.append(t2_time)
        print(f"  Two-step accuracy: {t2_acc:.4f}, time: {t2_time:.3f}s")

        gc.collect()
        gc.disable()
        tic = time.perf_counter()
        yPred1, XRecon1 = jointPredictReconstruct(XTestCensorFlat, model, C, mask)
        toc = time.perf_counter()
        gc.enable()
        t1_time = toc - tic
        t1_acc = np.sum(yPred1 == yTest) / yTest.size
        single_step_accs.append(t1_acc)
        single_step_times.append(t1_time)
        print(f"  Single-step accuracy: {t1_acc:.4f}, time: {t1_time:.3f}s")

        XRecon2 = reconstructImages(XTestCensorFlat[:18], yPred2[:18], model, mask)

        fig, axes = plt.subplots(3, 6, figsize=(12, 7))
        fig.suptitle(f'Reconstruction Comparison ({pct_label} missing)', fontsize=14)
        for j in range(6):
            axes[0, j].imshow(XTestCensorFlat[j].reshape(28, 28), cmap='gray', vmin=0, vmax=1)
            axes[0, j].set_title(f'True: {yTest[j]}', fontsize=8)
            axes[0, j].axis('off')
            axes[1, j].imshow(XRecon2[j].reshape(28, 28), cmap='gray', vmin=0, vmax=1)
            axes[1, j].set_title(f'2-step: {yPred2[j]}', fontsize=8)
            axes[1, j].axis('off')
            axes[2, j].imshow(XRecon1[j].reshape(28, 28), cmap='gray', vmin=0, vmax=1)
            axes[2, j].set_title(f'1-step: {yPred1[j]}', fontsize=8)
            axes[2, j].axis('off')
        axes[0, 0].set_ylabel('Censored', fontsize=10)
        axes[1, 0].set_ylabel('Two-step', fontsize=10)
        axes[2, 0].set_ylabel('Single-step', fontsize=10)
        plt.tight_layout()
        fig.savefig(f'recon_{start}_{end}.png', dpi=150, bbox_inches='tight')
        plt.close(fig)
        print(f"  Saved recon_{start}_{end}.png")

### Plot: Accuracy comparison

In [ ]:
missing_nums = [51, 32, 18, 8, 2]
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(missing_nums, two_step_accs, 'o-', label='Two-step (Lecture)', linewidth=2, markersize=8)
    ax.plot(missing_nums, single_step_accs, 's--', label='Single-step (Ours)', linewidth=2, markersize=8)
    ax.set_xlabel('Percentage of Missing Pixels (%)', fontsize=12)
    ax.set_ylabel('Classification Accuracy', fontsize=12)
    ax.set_title('Classification Accuracy vs Censoring Level', fontsize=13)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.invert_xaxis()
    plt.tight_layout()
    fig.savefig('accuracy_comparison.png', dpi=150, bbox_inches='tight')
    plt.close(fig)
    print("\nSaved accuracy_comparison.png")

### Plot: Inference time comparison

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(missing_nums, two_step_times, 'o-', label='Two-step (Lecture)', linewidth=2, markersize=8)
    ax.plot(missing_nums, single_step_times, 's--', label='Single-step (Ours)', linewidth=2, markersize=8)
    ax.set_xlabel('Percentage of Missing Pixels (%)', fontsize=12)
    ax.set_ylabel('Inference Time (seconds)', fontsize=12)
    ax.set_title('Inference Time vs Censoring Level', fontsize=13)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.invert_xaxis()
    plt.tight_layout()
    fig.savefig('time_comparison.png', dpi=150, bbox_inches='tight')
    plt.close(fig)
    print("Saved time_comparison.png")

    print("\n============================================")
    print("SUMMARY TABLE")
    print("============================================")
    print(f"{'Missing %':<12} {'2-Step Acc':<12} {'1-Step Acc':<12} {'2-Step Time':<12} {'1-Step Time':<12}")
    for i, pct in enumerate(missing_pcts):
        print(f"{pct:<12} {two_step_accs[i]:<12.4f} {single_step_accs[i]:<12.4f} {two_step_times[i]:<12.3f} {single_step_times[i]:<12.3f}")
    print("============================================")
    print("Done!")